In [1]:
!pip install transformers torch pdfplumber PyPDF2 python-igraph cairocffi gradio nltk sentence-transformers scikit-learn dbscan pca-multitask bertopic spacy
!python -m spacy download en_core_web_sm

  Using cached dbscan-1.0.0-cp39-abi3-macosx_11_0_arm64.whl.metadata (9.5 kB)
ERROR: Could not find a version that satisfies the requirement pca-multitask (from versions: none)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for pca-multitask
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 3.9 MB/s  0:00:03 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
!pip install bertopic spacy


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kakashi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import os
# FIX: Disable XET backend and progress bars to prevent 'shell_parent' LookupError
os.environ["HF_HUB_DISABLE_XET_BACKEND"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import gradio as gr
from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
import igraph as ig
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
import nltk
import re
import spacy
from spacy import displacy
from bertopic import BERTopic
from datetime import datetime

# Suppress warnings for a cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- GLOBAL MODEL/PIPELINE INITIALIZATION ---

# 1. NER Model for Knowledge Graph
MODEL_NAME = "CyberPeace-Institute/SecureBERT-NER"
NER_MODEL_INITIALIZED = False
ner_tokenizer = None
ner_pipeline = None

try:
    print("Attempting to load SecureBERT-NER Model...")
    ner_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    ner_model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
    ner_pipeline = pipeline(
        "token-classification",
        model=ner_model,
        tokenizer=ner_tokenizer,
        aggregation_strategy="simple"
    )
    print("NER Model loaded successfully.")
    NER_MODEL_INITIALIZED = True
except Exception as e:
    print(f"CRITICAL ERROR: Failed to load NER model. Knowledge Graph functionality will be disabled.")
    print(f"Details: {e}")

# 2. Sentence Embedding Model for Clustering
try:
    print("Attempting to load Sentence Transformer Model...")
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Sentence Transformer Model loaded successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Failed to load Sentence Transformer model. Clustering functionality will be disabled.")
    print(f"Details: {e}")

# 3. NLTK Tokenizer for Sentence Splitting
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    print("Downloading NLTK 'punkt' model...")
    nltk.download('punkt')

# 4. spaCy Model for Linguistic Analysis
try:
    print("Attempting to load spaCy Model...")
    nlp = spacy.load("en_core_web_sm")
    print("spaCy Model loaded successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Failed to load spaCy model: {e}")

# 5. Sentiment Analysis Model
sentiment_pipeline = None
try:
    print("Attempting to load Sentiment Model...")
    sentiment_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
    sentiment_pipeline = pipeline("sentiment-analysis", model=sentiment_model_name)
    print("Sentiment Pipeline loaded successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Failed to load Sentiment pipeline: {e}")

# --- CORE UTILITY FUNCTIONS ---

def extract_pdf_text(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + " \n"
        return text
    except Exception as e:
        return f"Error reading PDF file: {type(e).__name__}: {str(e)}"

def chunk_text(text, max_length=512, overlap=50):
    if not NER_MODEL_INITIALIZED: return ["Model not loaded."]
    tokens = ner_tokenizer.encode(text, add_special_tokens=False)
    chunks = [ner_tokenizer.decode(tokens[i:i + max_length]) for i in range(0, len(tokens), max_length - overlap)]
    return chunks

def clean_and_split_sentences(text):
    """
    Splits text into sentences and filters out noise for better clustering.
    """
    sentences = nltk.sent_tokenize(text)
    
    clean_sentences = []
    for sentence in sentences:
        sentence = re.sub(r'\s+', ' ', sentence).strip()
        word_count = len(sentence.split())
        if word_count < 4 or word_count > 256:
            continue
        if not re.search(r'[a-zA-Z]{3,}', sentence):
            continue
        if sentence.lower().startswith(("figure ", "table ", "page ", "©", "appendix ")):
            continue
        clean_sentences.append(sentence)
        
    return clean_sentences

def batch_sentiment_analysis(sentences):
    """
    Analyzes a list of sentences in a fast batch.
    """
    if not sentences:
        return pd.DataFrame(columns=["Label", "Score", "Sentence"]), "No sentences to analyze."
    if sentiment_pipeline is None:
        return pd.DataFrame(), "Sentiment pipeline not loaded."

    try:
        results = sentiment_pipeline(sentences, truncation=True)
        df = pd.DataFrame(results)
        valid_sentences = sentences[:len(df)]
        df['Sentence'] = valid_sentences
        df['Score'] = df['score'].round(3)
        df['Label'] = df['label']
        positive_df = df[df['Label'] == 'POSITIVE'].nlargest(5, 'Score')
        negative_df = df[df['Label'] == 'NEGATIVE'].nlargest(5, 'Score')
        summary_df = pd.concat([positive_df, negative_df]).sort_values('Score', ascending=False)
        return summary_df[['Label', 'Score', 'Sentence']], f"Analyzed {len(sentences)} sentences."
    except Exception as e:
        return pd.DataFrame(), f"Error during sentiment analysis: {e}"


def batch_cti_classification(sentences):
    """
    Scans a list of sentences for CTI keywords and returns a summary.
    """
    if not sentences:
        return pd.DataFrame(columns=["CTI Topic", "Mentions", "Example Sentence"]), "No sentences to analyze."

    keywords = {
        "Phishing": ["phishing", "vishing", "smishing"],
        "Malware": ["malware", "ransomware", "trojan", "keylogger", "emotet"],
        "Vulnerability": ["cve-", "vulnerability", "zero-day"],
        "Attack": ["attack", "breach", "incident", "apt-", "ddos"],
        "Exploit": ["exploit", "exploited", "rce", "remote code execution"],
    }
    topic_summary = {topic: {"count": 0, "example": ""} for topic in keywords}

    for sentence in sentences:
        sentence_lower = sentence.lower()
        found_in_sentence = set() 
        for topic, words in keywords.items():
            for word in words:
                if word in sentence_lower:
                    if topic not in found_in_sentence:
                        topic_summary[topic]["count"] += 1
                        if not topic_summary[topic]["example"]:
                            topic_summary[topic]["example"] = sentence
                        found_in_sentence.add(topic)
                        
    summary_list = []
    for topic, data in topic_summary.items():
        if data["count"] > 0:
            summary_list.append({
                "CTI Topic": topic,
                "Mentions": data["count"],
                "Example Sentence": data["example"]
            })
            
    if not summary_list:
        return pd.DataFrame([{"CTI Topic": "No CTI Keywords Found", "Mentions": 0, "Example Sentence": ""}]), "No CTI keywords found in document."

    summary_df = pd.DataFrame(summary_list).sort_values("Mentions", ascending=False)
    return summary_df, f"Scanned {len(sentences)} sentences for CTI terms."

# --- KNOWLEDGE GRAPH FUNCTIONS ---

def build_cti_knowledge_graph_igraph(entities, labels):
    name_to_original_label = {}
    vertex_names = []
    for ent, lab in zip(entities, labels):
        clean_ent = ent.replace('\n', ' ').strip()
        if clean_ent and clean_ent not in name_to_original_label:
            name_to_original_label[clean_ent] = lab
            vertex_names.append(clean_ent)
    
    G = ig.Graph(directed=True)
    G.add_vertices(len(vertex_names))
    G.vs["name"] = vertex_names
    G.vs["node_type"] = [name_to_original_label[name] for name in G.vs["name"]]
    G.vs["label"] = G.vs["name"]
    color_map = {'ACT':'#1f78b4','TOOL':'#33a02c','IDTY':'#ff7f00','TIME':'#cab2d6','MISC':'#a6cee3','APT':'#e31a1c','VULID':'#ffff99','IP':'#fdbf6f','URL':'#ff7f00','DOMAIN':'#b2df8a','FILE':'#fb9a99','HASH':'#a6cee3','CVE':'#ffff99','OS':'#cab2d6','PROTOCOL':'#fdbf6f'}
    G.vs["color"] = [color_map.get(lab, '#a6cee3') for lab in G.vs["node_type"]]
    
    edges_to_add = []
    edge_relations = []
    cleaned_entities = [ent.replace('\n', ' ').strip() for ent in entities]
    
    for i in range(len(cleaned_entities) - 1):
        e1, l1 = cleaned_entities[i], labels[i]
        e2, l2 = cleaned_entities[i+1], labels[i+1]
        if not e1 or not e2 or e1 not in G.vs["name"] or e2 not in G.vs["name"]: continue
        id1 = G.vs.find(name=e1).index
        id2 = G.vs.find(name=e2).index
        relation = "related_to"
        if l1 == "IDTY" and l2 == "ACT": relation = "performs_ttp"
        elif l1 == "ACT" and l2 == "TOOL": relation = "uses_tool"
        elif l1 == "APT" and l2 == "MALWARE": relation = "uses_malware"
        elif l1 == "MALWARE" and l2 in ["IP", "URL", "DOMAIN"]: relation = "communicates_with"
        elif l1 == "VULID" and l2 in ["OS", "TOOL"]: relation = "affects"
        edges_to_add.append((id1, id2))
        edge_relations.append(relation)
        
    G.add_edges(edges_to_add)
    G.es["label"] = edge_relations
    return G

def query_entity_graph_igraph(G, entity_name):
    if G is None or entity_name is None: return None, "Graph not generated. Please process a report first."
    clean_name = entity_name.replace('\n', ' ').strip()
    if clean_name not in G.vs["name"]: return None, f"Entity '{clean_name}' not found."
    try:
        center_vid = G.vs.find(name=clean_name).index
        neighbor_vids = G.neighbors(center_vid, mode="all")
        subgraph = G.induced_subgraph(list(set([center_vid] + neighbor_vids)))
        if not subgraph.vs: return None, f"Entity '{clean_name}' has no connections to plot."
        layout = subgraph.layout("kamada_kawai")
        visual_style = {"vertex_label": subgraph.vs["name"], "vertex_color": subgraph.vs["color"], "edge_label": subgraph.es["label"], "edge_color": "gray", "vertex_size": 25, "vertex_label_size": 10, "edge_label_size": 9, "bbox": (800, 600), "margin": 50}
        fig, ax = plt.subplots(figsize=(10, 8))
        ig.plot(subgraph, target=ax, layout=layout, **visual_style)
        ax.set_title(f"Knowledge Graph: 1-Hop Neighbors of '{clean_name}'")
        return fig, f"Successfully mapped {subgraph.vcount()} connections."
    except Exception as e:
        plt.close('all')
        return None, f"Error generating subgraph: {e}"

# --- SEMANTIC CLUSTERING & TOPIC MODELING FUNCTIONS ---

def get_cluster_topic_names(sentences, cluster_assignments):
    clustered_sentences = {i: [] for i in set(cluster_assignments)}
    for sentence, cluster_id in zip(sentences, cluster_assignments):
        clustered_sentences[cluster_id].append(sentence)
    topic_names = {}
    for cluster_id, docs in clustered_sentences.items():
        if cluster_id == -1:
            topic_names[cluster_id] = "Outliers / Miscellaneous"
            continue
        try:
            vectorizer = TfidfVectorizer(stop_words='english', max_features=3, ngram_range=(1, 2))
            corpus = [" ".join(docs)]
            vectorizer.fit(corpus)
            feature_names = vectorizer.get_feature_names_out()
            topic_names[cluster_id] = ", ".join(feature_names)
        except ValueError: 
            topic_names[cluster_id] = "Short / Common Phrases"
    return topic_names

def perform_clustering(sentences):
    if not sentences: return None, None, None, "No sentences to cluster."
    embeddings = embedding_model.encode(sentences)
    dbscan = DBSCAN(eps=1.0, min_samples=2)
    dbscan.fit(embeddings)
    cluster_assignments = dbscan.labels_
    topic_names = get_cluster_topic_names(sentences, cluster_assignments)
    return embeddings, cluster_assignments, topic_names, f"Successfully clustered {len(sentences)} sentences."

def create_cluster_plot(embeddings, cluster_assignments, topic_names):
    if embeddings is None: return None
    pca = PCA(n_components=2)
    reduced_embeddings = pca.fit_transform(embeddings)
    fig, ax = plt.subplots(figsize=(12, 10))
    unique_labels = sorted(set(cluster_assignments))
    colors = [plt.cm.viridis(each) for each in np.linspace(0, 1, len(unique_labels))]
    for k, col in zip(unique_labels, colors):
        label = topic_names.get(k, "Unknown")
        if k == -1: col = [0, 0, 0, 1]
        class_member_mask = (cluster_assignments == k)
        xy = reduced_embeddings[class_member_mask]
        ax.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                markeredgecolor='k', markersize=14 if k != -1 else 7, label=label)
    ax.set_title("Semantic Topic Clusters from PDF Document")
    ax.legend(title="Topics")
    return fig

def show_cluster_sentences(selected_topic, topics_dict, assignments_list, sentences_list):
    if not selected_topic:
        return pd.DataFrame(columns=["Sentences"]), "Select a topic to see sample sentences."
    try:
        cluster_id = [key for key, value in topics_dict.items() if value == selected_topic][0]
        matching_sentences = []
        for sentence, assignment in zip(sentences_list, assignments_list):
            if assignment == cluster_id:
                matching_sentences.append(sentence)
        df = pd.DataFrame(matching_sentences, columns=["Sentences"])
        status = f"Showing {len(matching_sentences)} sentences for topic: '{selected_topic}'"
        return df, status
    except Exception as e:
        return pd.DataFrame(), f"Error finding sentences: {e}"

# --- NEW: BERTopic Function ---
def run_bertopic_modeling(sentences):
    if not sentences:
        return None, None, "No sentences to model. Please process a report first."
    
    try:
        print("Starting BERTopic modeling...")
        topic_model = BERTopic(verbose=False, min_topic_size=2, embedding_model=embedding_model)
        topics, probs = topic_model.fit_transform(sentences)
        
        # Get topic info for a table
        topic_info = topic_model.get_topic_info()
        
        # Get the barchart
        fig = topic_model.visualize_barchart(top_n_topics=10)
        
        print("BERTopic modeling complete.")
        return fig, topic_info, "BERTopic analysis complete."
    except Exception as e:
        return None, None, f"Error during BERTopic analysis: {e}"


# --- NEW: Linguistic Analysis Function ---

def linguistic_analysis_spacy(text):
    if not text or not text.strip():
        return [], "<p>Please enter text for analysis.</p>"
    
    doc = nlp(text) # Uses the global nlp model
    pos_tags = [(t.text, t.pos_, t.dep_) for t in doc]

    # Generate the raw SVG from displacy, ensuring text is dark
    options = {"color": "#000000", "font": "sans-serif"}
    svg = displacy.render(doc, style="dep", jupyter=False, options=options) 
    
   
    html_wrapper = f"""
    <div style="background-color: white; border: 1px solid #E5E7EB; border-radius: 8px; padding: 12px; overflow-x: auto;">
        {svg}
    </div>
    """
    
    return pos_tags, html_wrapper
    
    
    
# --- GRADIO WORKFLOW FUNCTIONS ---

def unified_process_report(file_obj):
    initial_df = pd.DataFrame(columns=['Entity', 'Type', 'Score'])
    initial_dropdown = gr.Dropdown(choices=[], value=None)
    if file_obj is None: return initial_df, initial_dropdown, "Please upload a PDF file.", [], None
    if not NER_MODEL_INITIALIZED: return initial_df, initial_dropdown, "CRITICAL: NER Model failed to load.", [], None
    text = extract_pdf_text(file_obj.name)
    if text.startswith("Error"): return initial_df, initial_dropdown, text, [], None
    sentences = clean_and_split_sentences(text)
    chunks = chunk_text(text)
    results = [res for chunk in chunks for res in ner_pipeline(chunk)]

    if not results:
        status = f"Processed. Found {len(sentences)} clean sentences. NER found no entities."
        return initial_df, initial_dropdown, status, sentences, None
    df = pd.DataFrame(results).rename(columns={'word': 'Entity', 'entity_group': 'Type'})
    df['Score'] = df['score'].round(4)
    df_display = df[['Entity', 'Type', 'Score']].copy()
    try:
        G = build_cti_knowledge_graph_igraph(df["Entity"].tolist(), df["Type"].tolist())
        unique_entity_names = G.vs["name"]
        final_status = f"Processed. Found {len(sentences)} clean sentences and {G.vcount()} unique entities."
        return df_display, gr.Dropdown(choices=unique_entity_names, value=None), final_status, sentences, G
    except Exception as e:
        return initial_df, initial_dropdown, f"Error building graph: {e}", sentences, None

def run_clustering_workflow(sentences):
    embeddings, labels, topics, status = perform_clustering(sentences)
    plot = create_cluster_plot(embeddings, labels, topics)
    topic_name_list = list(topics.values())
    sentence_df = pd.DataFrame(sentences, columns=["Sentences"])
    return plot, status, labels, topics, gr.Dropdown(choices=topic_name_list), sentence_df

def run_batch_analysis(sentences):
    cti_df, cti_status = batch_cti_classification(sentences)
    sent_df, sent_status = batch_sentiment_analysis(sentences)
    full_status = f"CTI: {cti_status} | Sentiment: {sent_status}"
    return cti_df, sent_df, full_status

# --- GRADIO INTERFACE LAYOUT ---

with gr.Blocks(title="CTI Analysis Tool", theme=gr.themes.Soft()) as app:
    gr.Markdown("# Cyber Threat Intelligence (CTI) Analysis Tool")
    gr.Markdown("Upload a CTI report (PDF) to analyze entities and semantic topics.")
    
    # --- State Variables ---
    sentences_state = gr.State([])
    graph_state = gr.State(None)
    cluster_assignments_state = gr.State([]) 
    cluster_topics_state = gr.State({})

    # --- Main Upload Row ---
    with gr.Row():
        file_input = gr.File(label="Upload CTI Report (PDF)", file_types=[".pdf"])
        process_button = gr.Button("Process Report", variant="primary")
    status_output = gr.Textbox(label="Processing Status", interactive=False)

    # --- Tabs ---
    with gr.Tabs():
        with gr.TabItem("Knowledge Graph Analyzer"):
            gr.Markdown("### Visualize Entities and Their Relationships")
            entity_table_output = gr.DataFrame(headers=["Entity", "Type", "Score"], label="Extracted Entities")
            with gr.Row():
                entity_dropdown = gr.Dropdown(label="Select an Entity to Query", choices=[], interactive=True, scale=2)
                query_button = gr.Button("Show Subgraph", scale=1)
            graph_output = gr.Plot(label="1-Hop Knowledge Graph Subgraph")
            graph_status = gr.Textbox(label="Graph Status", interactive=False)

        with gr.TabItem("Semantic Topic Clustering"):
            gr.Markdown("### Group Sentences by Semantic Meaning (DBSCAN)")
            cluster_button = gr.Button("1. Cluster PDF Sentences", variant="secondary")
            cluster_status = gr.Textbox(label="Clustering Status", interactive=False)
            gr.Markdown("#### Sentences Used for Clustering")
            input_sentence_df = gr.DataFrame(headers=["Sentences"], label="Input Sentences", interactive=False, row_count=10)
            gr.Markdown("#### Cluster Visualization")
            cluster_plot_output = gr.Plot(label="Sentence Cluster Visualization")
            gr.Markdown("### Explore Clusters")
            with gr.Row():
                topic_dropdown = gr.Dropdown(label="Select Topic", choices=[], interactive=True, scale=3)
                cluster_sentence_df = gr.DataFrame(headers=["Sentences"], label="Sentences in Selected Cluster", interactive=False, scale=4, row_count=10)
            
        with gr.TabItem("Document Summary"):
            gr.Markdown("### Sentiment & CTI Summary")
            analyze_pdf_button = gr.Button("Analyze PDF Sentences", variant="primary")
            summary_status = gr.Textbox(label="Analysis Status", interactive=False)
            gr.Markdown("#### CTI Keyword Summary")
            cti_summary_output = gr.DataFrame(headers=["CTI Topic", "Mentions", "Example Sentence"], label="CTI Summary")
            gr.Markdown("#### Sentiment Analysis")
            sentiment_summary_output = gr.DataFrame(headers=["Label", "Score", "Sentence"], label="Sentiment Highlights", row_count=10)

        # --- NEW: BERTopic Tab ---
        with gr.TabItem("Topic Modeling (BERTopic)"):
            gr.Markdown("### Advanced Topic Modeling with BERTopic")
            gr.Markdown("Run BERTopic on the full list of cleaned sentences to discover themes.")
            bertopic_button = gr.Button("Run Topic Model", variant="secondary")
            bertopic_status = gr.Textbox(label="BERTopic Status", interactive=False)
            gr.Markdown("#### Top 10 Discovered Topics")
            bertopic_plot = gr.Plot(label="BERTopic Barchart")
            gr.Markdown("#### All Discovered Topics")
            bertopic_df = gr.DataFrame(label="BERTopic Topic List")

        # --- NEW: Linguistic Analysis Tab ---
        with gr.TabItem("Linguistic Analysis (spaCy)"):
            gr.Markdown("### POS Tagging & Dependency Parsing")
            gr.Markdown("Analyze the grammatical structure of a single sentence.")
            ling_input = gr.Textbox(label="Enter a sentence to analyze", lines=3, placeholder="e.g., Copy a sentence from the cluster results...")
            ling_button = gr.Button("Analyze Syntax")
            gr.Markdown("#### Part-of-Speech (POS) Tags")
            ling_pos_df = gr.DataFrame(headers=["Token", "POS", "Dependency"], label="POS Tags", row_count=10)
            gr.Markdown("#### Dependency Plot")
            ling_dep_html = gr.HTML(label="Dependency Visualization")

    # --- EVENT HANDLERS ---
    process_button.click(
        fn=unified_process_report,
        inputs=[file_input],
        outputs=[entity_table_output, entity_dropdown, status_output, sentences_state, graph_state]
    )

    query_button.click(
        fn=query_entity_graph_igraph,
        inputs=[graph_state, entity_dropdown],
        outputs=[graph_output, graph_status]
    )
    
    entity_dropdown.select(
        fn=query_entity_graph_igraph,
        inputs=[graph_state, entity_dropdown],
        outputs=[graph_output, graph_status]
    )

    cluster_button.click(
        fn=run_clustering_workflow,
        inputs=[sentences_state],
        outputs=[
            cluster_plot_output, 
            cluster_status, 
            cluster_assignments_state,
            cluster_topics_state,
            topic_dropdown,
            input_sentence_df 
        ]
    )
    
    topic_dropdown.select(
        fn=show_cluster_sentences,
        inputs=[
            topic_dropdown, 
            cluster_topics_state, 
            cluster_assignments_state, 
            sentences_state
        ],
        outputs=[cluster_sentence_df, cluster_status]
    )
    
    analyze_pdf_button.click(
        fn=run_batch_analysis,
        inputs=sentences_state, 
        outputs=[cti_summary_output, sentiment_summary_output, summary_status]
    )
    
    # --- NEW: Event Handlers for New Tabs ---
    bertopic_button.click(
        fn=run_bertopic_modeling,
        inputs=[sentences_state],
        outputs=[bertopic_plot, bertopic_df, bertopic_status]
    )
    
    ling_button.click(
        fn=linguistic_analysis_spacy,
        inputs=[ling_input],
        outputs=[ling_pos_df, ling_dep_html]
    )

app.launch(debug=True)

Attempting to load SecureBERT-NER Model...


Device set to use mps:0


NER Model loaded successfully.
Attempting to load Sentence Transformer Model...
Sentence Transformer Model loaded successfully.
Attempting to load spaCy Model...
spaCy Model loaded successfully.
Attempting to load Sentiment Model...


Device set to use mps:0


Sentiment Pipeline loaded successfully.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Token indices sequence length is longer than the specified maximum sequence length for this model (30627 > 512). Running this sequence through the model will result in indexing errors


Keyboard interruption in main thread... closing server.
